In [7]:
# !pip install cdsapi
# !pip install eccodes
# !pip install cfgrib eccodes
# !conda install -c conda-forge eccodes cfgrib xarray -y

### API Inwersja - Poznań 2024

In [1]:
import cdsapi

dataset = "reanalysis-era5-pressure-levels"
request = {
    "product_type": ["reanalysis"],
    "variable": ["temperature"],
    "year": ["2024"],
    "month": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12"
    ],
    "day": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28", "29", "30",
        "31"
    ],
    "time": [
        "00:00", "06:00", "12:00",
        "18:00"
    ],
    "pressure_level": ["925", "1000"],
    "data_format": "grib",
    "download_format": "unarchived",
    "area": [52.54000, 16.80000, 52.28000, 17.06000]
}

client = cdsapi.Client(
    url = "https://cds.climate.copernicus.eu/api",
    key = "f55a5040-c43c-40a7-957a-f078b44210f3"
)
client.retrieve(dataset, request).download()

2026-04-04 16:58:22,619 INFO Request ID is 9493a5d2-40dd-4c9a-8252-509ed2635b5f
2026-04-04 16:58:22,748 INFO status has been updated to accepted
2026-04-04 16:58:36,689 INFO status has been updated to successful


ed3a13c17a341488faa4717e8df1eab0.grib:   0%|          | 0.00/315k [00:00<?, ?B/s]

'ed3a13c17a341488faa4717e8df1eab0.grib'

In [1]:
import eccodes
import xarray as xr
import cfgrib

# Odczyt pliku GRIB przez xarray
ds = xr.open_dataset(
    'ed3a13c17a341488faa4717e8df1eab0.grib',
    engine='cfgrib'
)

Can't create file 'ed3a13c17a341488faa4717e8df1eab0.grib.5b7b6.idx'
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/site-packages/cfgrib/messages.py", line 539, in from_indexpath_or_filestream
    self = cls.from_fieldset(filestream, index_keys, computed_keys)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/cfgrib/messages.py", line 379, in from_fieldset
    return cls.from_fieldset_and_iteritems(fieldset, iteritems, index_keys, computed_keys)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/cfgrib/messages.py", line 392, in from_fieldset_and_iteritems
    for field_id, raw_field in iteritems:
  File "/opt/anaconda3/lib/python3.12/site-packages/cfgrib/messages.py", line 292, in __iter__
    for message in self.itervalues():
  File "/opt/anaconda3/lib/python3.12/site-packages/cfgrib/messages.py", li

FileNotFoundError: [Errno 2] No such file or directory: 'ed3a13c17a341488faa4717e8df1eab0.grib'

In [7]:
import pandas as pd

df = ds.to_dataframe().reset_index()


# Wyświetl wszystkie wiersze i kolumny
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print(df.shape)  # ile wierszy x kolumn
df.head()

(2928, 8)


,time,isobaricInhPa,latitude,longitude,number,step,valid_time,t
0,2024-01-01 00:00:00,1000.0,52.5,17.0,0,0 days,2024-01-01 00:00:00,279.217041
1,2024-01-01 00:00:00,925.0,52.5,17.0,0,0 days,2024-01-01 00:00:00,277.268555
2,2024-01-01 06:00:00,1000.0,52.5,17.0,0,0 days,2024-01-01 06:00:00,279.074219
3,2024-01-01 06:00:00,925.0,52.5,17.0,0,0 days,2024-01-01 06:00:00,276.662842
4,2024-01-01 12:00:00,1000.0,52.5,17.0,0,0 days,2024-01-01 12:00:00,279.577881


In [9]:
# Rozdzielam na dwa DataFrame 
df_925 = df[df['isobaricInhPa'] == 925][['time','t']].rename(columns = {'t': 't_925'})
df_1000 = df[df['isobaricInhPa'] == 1000][['time','t']].rename(columns = {'t': 't_1000'})

# Łaczymy oba df, w jeden do obliczenia różnicy
df_delta = pd.merge(df_925, df_1000, how='inner', on='time')

# kolumna t_delta = podaje rożnice w stopniach 
df_delta['t_delta'] = df_delta['t_1000']-df_delta['t_925']
df_delta['inversion_0/1'] = [1 if e > 0 else 0 for e in df_delta['t_delta']]

# wyciagamy sama date , do łatiwejszegeo grupowania i wyciagniecia dziennej sreniej 

df_delta['data'] = df_delta['time'].dt.date
df_delta.tail()

,time,t_925,t_1000,t_delta,inversion_0/1,data
1459,2024-12-30 18:00:00,273.469238,274.429688,0.960449,1,2024-12-30
1460,2024-12-31 00:00:00,277.173340,274.001465,-3.171875,0,2024-12-31
1461,2024-12-31 06:00:00,279.160645,274.318848,-4.841797,0,2024-12-31
1462,2024-12-31 12:00:00,280.300537,274.890137,-5.410400,0,2024-12-31
1463,2024-12-31 18:00:00,277.722168,274.690918,-3.031250,0,2024-12-31


In [11]:
# df_delta.to_csv('/Users/marcinurbanski/Desktop/merito_projekt_pliki_robocze/qualityair_projekt/ready_raw_data/inwersja_pz_2024.csv')

In [9]:
# import pandas as pd

# # Konwersja do DataFrame (temperatura w Kelvinach -> Celsjusze)
# df = ds['t'].to_dataframe().reset_index()
# df['temperature_C'] = df['t'] - 273.15

# print(df.head(20))
# print(f"\nZakres dat: {df['time'].min()} → {df['time'].max()}")
# print(f"Poziomy ciśnienia: {df['isobaricInhPa'].unique()}")

                  time  isobaricInhPa  number   step  latitude  longitude  \
0  2024-01-01 00:00:00         1000.0       0 0 days      53.5       14.5   
1  2024-01-01 00:00:00          925.0       0 0 days      53.5       14.5   
2  2024-01-01 06:00:00         1000.0       0 0 days      53.5       14.5   
3  2024-01-01 06:00:00          925.0       0 0 days      53.5       14.5   
4  2024-01-01 12:00:00         1000.0       0 0 days      53.5       14.5   
5  2024-01-01 12:00:00          925.0       0 0 days      53.5       14.5   
6  2024-01-01 18:00:00         1000.0       0 0 days      53.5       14.5   
7  2024-01-01 18:00:00          925.0       0 0 days      53.5       14.5   
8  2024-01-02 00:00:00         1000.0       0 0 days      53.5       14.5   
9  2024-01-02 00:00:00          925.0       0 0 days      53.5       14.5   
10 2024-01-02 06:00:00         1000.0       0 0 days      53.5       14.5   
11 2024-01-02 06:00:00          925.0       0 0 days      53.5       14.5   

##@ API INWERSJA - ŁÓDŹ 2004

In [5]:
import cdsapi

dataset = "reanalysis-era5-pressure-levels"
request = {
    "product_type": ["reanalysis"],
    "variable": ["temperature"],
    "year": ["2004"],
    "month": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12"
    ],
    "day": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28", "29", "30",
        "31"
    ],
    "time": [
        "00:00", "06:00", "12:00",
        "18:00"
    ],
    "pressure_level": ["925", "1000"],
    "data_format": "grib",
    "download_format": "unarchived",
    "area": [51.91000, 19.33000, 51.64000, 19.59000]
}

client = cdsapi.Client(
    url = "https://cds.climate.copernicus.eu/api",
    key = "f55a5040-c43c-40a7-957a-f078b44210f3"
)
client.retrieve(dataset, request).download()


2026-04-09 18:41:59,608 INFO Request ID is ced2b088-757e-487d-bcdb-bda71d7304d0
2026-04-09 18:41:59,738 INFO status has been updated to accepted
2026-04-09 18:50:21,447 INFO status has been updated to running
2026-04-09 18:56:23,067 INFO status has been updated to successful


4b287d4118ad2f31e78bc1585293f106.grib:   0%|          | 0.00/315k [00:00<?, ?B/s]

'4b287d4118ad2f31e78bc1585293f106.grib'

In [27]:
import eccodes
import xarray as xr
import cfgrib

# Odczyt pliku GRIB przez xarray
ds_inwersja_ld_2004 = xr.open_dataset(
    '4b287d4118ad2f31e78bc1585293f106.grib',
    engine='cfgrib'
)

import pandas as pd
df_inwersja_ld_2004 = ds_inwersja_ld_2004.to_dataframe().reset_index()

# Wyświetl wszystkie wiersze i kolumny
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# print(df_inwersja_ld_2004.shape)  # ile wierszy x kolumn
# df_inwersja_ld_2004[['time'=2004-02-20]]


Timestamp('2004-01-01 00:00:00')

In [63]:
df_inwersja_ld_2004_delta.loc[df_inwersja_ld_2004['time']=='2004-02-20 00:00:00']

,time,isobaricInhPa,latitude,longitude,number,step,valid_time,t
400,2004-02-20,1000.0,51.75,19.5,0,0 days,2004-02-20,263.837891
401,2004-02-20,925.0,51.75,19.5,0,0 days,2004-02-20,264.801758


In [65]:
# Rozdzielam na dwa DataFrame 
df_inwersja_ld_2004_925 = df_inwersja_ld_2004[df_inwersja_ld_2004['isobaricInhPa'] == 925][['time','t']].rename(columns = {'t': 't_925'})
df_inwersja_ld_2004_1000 = df_inwersja_ld_2004[df_inwersja_ld_2004['isobaricInhPa'] == 1000][['time','t']].rename(columns = {'t': 't_1000'})

# Łaczymy oba df, w jeden do obliczenia różnicy
df_inwersja_ld_2004_delta = pd.merge(df_inwersja_ld_2004_925, df_inwersja_ld_2004_1000, how='inner', on='time')

# kolumna t_delta = podaje rożnice w stopniach 
df_inwersja_ld_2004_delta['t_delta'] = df_inwersja_ld_2004_delta['t_1000']-df_inwersja_ld_2004_delta['t_925']
df_inwersja_ld_2004_delta['inversion_0/1'] = [1 if e > 0 else 0 for e in df_inwersja_ld_2004_delta['t_delta']]

# wyciagamy sama date , do łatiwejszegeo grupowania i wyciagniecia dziennej sreniej 

df_inwersja_ld_2004_delta['data'] = df_inwersja_ld_2004_delta['time'].dt.date
df_inwersja_ld_2004_delta['city'] = 'ld'
df_inwersja_ld_2004_delta.head(15)

,time,t_925,t_1000,t_delta,inversion_0/1,data,city
0,2004-01-01 00:00:00,268.939941,272.321533,3.381592,1,2004-01-01,ld
1,2004-01-01 06:00:00,268.319092,271.235596,2.916504,1,2004-01-01,ld
2,2004-01-01 12:00:00,267.282715,272.199707,4.916992,1,2004-01-01,ld
3,2004-01-01 18:00:00,266.759277,270.886230,4.126953,1,2004-01-01,ld
4,2004-01-02 00:00:00,266.369141,270.640381,4.271240,1,2004-01-02,ld
5,2004-01-02 06:00:00,264.989746,269.482910,4.493164,1,2004-01-02,ld
6,2004-01-02 12:00:00,264.485596,269.442383,4.956787,1,2004-01-02,ld
7,2004-01-02 18:00:00,262.840332,267.762939,4.922607,1,2004-01-02,ld
8,2004-01-03 00:00:00,260.855713,265.802246,4.946533,1,2004-01-03,ld
9,2004-01-03 06:00:00,259.543213,264.646973,5.103760,1,2004-01-03,ld


In [67]:
df_inwersja_ld_2004_delta.to_csv('/Users/marcinurbanski/Desktop/merito_projekt_pliki_robocze/qualityair_projekt/0_ready_raw_data/inwersja_ld_2004.csv')


### API INWERSJA - ŁÓDŹ 2024

In [6]:
import cdsapi

dataset = "reanalysis-era5-pressure-levels"
request = {
    "product_type": ["reanalysis"],
    "variable": ["temperature"],
    "year": ["2024"],
    "month": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12"
    ],
    "day": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28", "29", "30",
        "31"
    ],
    "time": [
        "00:00", "06:00", "12:00",
        "18:00"
    ],
    "pressure_level": ["925", "1000"],
    "data_format": "grib",
    "download_format": "unarchived",
    "area": [51.91000, 19.33000, 51.64000, 19.59000]
}

client = cdsapi.Client(
    url = "https://cds.climate.copernicus.eu/api",
    key = "f55a5040-c43c-40a7-957a-f078b44210f3"
)
client.retrieve(dataset, request).download()

2026-04-09 18:56:25,174 INFO Request ID is ca243b6b-d8f2-436f-a479-627a92072d4e
2026-04-09 18:56:25,274 INFO status has been updated to accepted
2026-04-09 19:04:46,670 INFO status has been updated to running
2026-04-09 19:08:47,780 INFO status has been updated to successful


ea8be334f68d71b85bef2da9a8803572.grib:   0%|          | 0.00/315k [00:00<?, ?B/s]

'ea8be334f68d71b85bef2da9a8803572.grib'

In [15]:
import eccodes
import xarray as xr
import cfgrib

# Odczyt pliku GRIB przez xarray
ds_inwersja_ld_2024 = xr.open_dataset(
    'ea8be334f68d71b85bef2da9a8803572.grib',
    engine='cfgrib'
)

import pandas as pd
df_inwersja_ld_2024 = ds_inwersja_ld_2024.to_dataframe().reset_index()

# Wyświetl wszystkie wiersze i kolumny
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print(df_inwersja_ld_2024.shape)  # ile wierszy x kolumn
df_inwersja_ld_2024.head()

(2928, 8)


,time,isobaricInhPa,latitude,longitude,number,step,valid_time,t
0,2024-01-01 00:00:00,1000.0,51.75,19.5,0,0 days,2024-01-01 00:00:00,280.791260
1,2024-01-01 00:00:00,925.0,51.75,19.5,0,0 days,2024-01-01 00:00:00,277.645508
2,2024-01-01 06:00:00,1000.0,51.75,19.5,0,0 days,2024-01-01 06:00:00,279.488281
3,2024-01-01 06:00:00,925.0,51.75,19.5,0,0 days,2024-01-01 06:00:00,276.463623
4,2024-01-01 12:00:00,1000.0,51.75,19.5,0,0 days,2024-01-01 12:00:00,278.585693


In [71]:
# Rozdzielam na dwa DataFrame 
df_inwersja_ld_2024_925 = df_inwersja_ld_2024[df_inwersja_ld_2024['isobaricInhPa'] == 925][['time','t']].rename(columns = {'t': 't_925'})
df_inwersja_ld_2024_1000 = df_inwersja_ld_2024[df_inwersja_ld_2024['isobaricInhPa'] == 1000][['time','t']].rename(columns = {'t': 't_1000'})

# Łaczymy oba df, w jeden do obliczenia różnicy
df_inwersja_ld_2024_delta = pd.merge(df_inwersja_ld_2024_925, df_inwersja_ld_2024_1000, how='inner', on='time')

# kolumna t_delta = podaje rożnice w stopniach 
df_inwersja_ld_2024_delta['t_delta'] = df_inwersja_ld_2024_delta['t_1000']-df_inwersja_ld_2024_delta['t_925']
df_inwersja_ld_2024_delta['inversion_0/1'] = [1 if e > 0 else 0 for e in df_inwersja_ld_2024_delta['t_delta']]

# wyciagamy sama date , do łatiwejszegeo grupowania i wyciagniecia dziennej sreniej 

df_inwersja_ld_2024_delta['data'] = df_inwersja_ld_2024_delta['time'].dt.date
df_inwersja_ld_2024_delta['city'] = 'ld'
df_inwersja_ld_2024_delta.head()

,time,t_925,t_1000,t_delta,inversion_0/1,data,city
0,2024-01-01 00:00:00,277.645508,280.791260,3.145752,1,2024-01-01,ld
1,2024-01-01 06:00:00,276.463623,279.488281,3.024658,1,2024-01-01,ld
2,2024-01-01 12:00:00,274.171387,278.585693,4.414307,1,2024-01-01,ld
3,2024-01-01 18:00:00,275.126465,278.893555,3.767090,1,2024-01-01,ld
4,2024-01-02 00:00:00,274.584473,278.025635,3.441162,1,2024-01-02,ld


In [73]:
# df_inwersja_ld_2024_delta.to_csv('/Users/marcinurbanski/Desktop/merito_projekt_pliki_robocze/qualityair_projekt/0_ready_raw_data/inwersja_ld_2024.csv')